# Extraction Pipeline — Test Notebook

Smoke-tests each collector in the `extract` package and inspects what lands in SQLite.

**Order of operations**
1. Setup & imports
2. Initialize the database
3. IRS 990 parser (no API key needed — run this first)
4. Congress.gov (needs `CONGRESS_API_KEY`)
5. FEC openFEC (works with `DEMO_KEY`, slowly)
6. Senate LDA (works anonymously)
7. Inspect the database

> Keys are read from `.env`. Cells for sources without a key will skip gracefully.

In [ ]:
# 1. Setup — make the project root importable and enable logging.
# The notebook lives in notebooks/, so add the parent dir to sys.path.
import logging
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")

from extract.config import settings

print("Project root:", PROJECT_ROOT)
print("Database path:", settings.db_path)
print("Keys present -> congress:", bool(settings.congress_api_key),
      "| fec:", settings.fec_api_key or "(none)",
      "| lda:", bool(settings.lda_api_key))

## 2. Initialize the database

Creates `data/influence.db` with all tables if it doesn't exist yet. Safe to re-run.

In [ ]:
# 2. Initialize the schema.
from extract.db import init_db

init_db()
print("Database initialized at", settings.db_path)

## 3. IRS Form 990 parser

No API key required. Parses the bundled fixture, then ingests it into SQLite.
Point `ingest_990_directory` at a real folder of 990 XML files when you have them.

In [ ]:
# 3a. Parse the sample 990 fixture and inspect the structured output.
import json

from extract.irs990 import parse_990_file

fixture = PROJECT_ROOT / "tests" / "fixtures" / "sample_990.xml"
parsed = parse_990_file(fixture)
print(json.dumps(parsed, indent=2))

In [ ]:
# 3b. Ingest the whole fixtures folder into SQLite.
# For real data: ingest_990_directory("data/irs990_xml")
from extract.irs990 import ingest_990_directory

n_files = ingest_990_directory(PROJECT_ROOT / "tests" / "fixtures")
print(f"Ingested {n_files} 990 file(s).")

## 4. Congress.gov collector

Needs `CONGRESS_API_KEY` in `.env`. The `limit` keeps this a quick smoke test —
each bill triggers several sub-requests (actions, cosponsors, text versions).

In [ ]:
# 4. Collect a handful of 118th Congress House bills.
if settings.congress_api_key:
    from extract.congress import CongressCollector

    n_bills = CongressCollector().collect_bills(congress=118, bill_type="hr", limit=5)
    print(f"Collected {n_bills} bills.")
else:
    print("SKIPPED: set CONGRESS_API_KEY in .env to run this cell.")

## 5. FEC openFEC collector

`DEMO_KEY` works but is heavily rate-limited — register a free key for real pulls.
Committee type `O` = independent-expenditure-only (Super PAC).

In [ ]:
# 5. Collect some Super PAC committees + a few disbursements.
from extract.fec import FecCollector

fec = FecCollector()
n_committees = fec.collect_committees(committee_type="O", limit=25)
print(f"Collected {n_committees} committees.")

n_disb = fec.collect_disbursements(cycle=2024, limit=25)
print(f"Collected {n_disb} disbursements.")

## 6. Senate LDA collector

Works anonymously (slower). Pulls lobbying filings plus their free-text issue
descriptions — the input for the NLP alignment work later.

In [ ]:
# 6. Collect a few 2024 lobbying filings.
from extract.lda import LdaCollector

n_filings = LdaCollector().collect_filings(filing_year=2024, limit=10)
print(f"Collected {n_filings} LDA filings.")

## 7. Inspect the database

Row counts per table, then a peek at a couple of tables with pandas.

In [ ]:
# 7a. Row counts across every table.
from extract.db import connect

tables = [
    "bills", "bill_actions", "bill_sponsors", "bill_text_versions", "members",
    "committees", "fec_disbursements",
    "lda_filings", "lda_lobbying_activities",
    "orgs", "org_grants", "org_people",
]
with connect() as conn:
    for t in tables:
        n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"{t:<24} {n:>6}")

In [ ]:
# 7b. Load a few tables into pandas for a closer look.
import pandas as pd

with connect() as conn:
    orgs = pd.read_sql_query("SELECT ein, name, total_revenue, political_activity_flag FROM orgs", conn)
    grants = pd.read_sql_query("SELECT grantor_ein, grantee_ein, grantee_name, amount FROM org_grants", conn)

display(orgs)
display(grants)